# demo for parallel nodes execution

Start <br>
-> calcualte bonus<br>
-> calcaualte yearly salary<br>
-> project eval<br>
Summary<br><br>
END<br>

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-20b")

# non LLM flow

In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal

class EmployeeDetails(TypedDict):
    name : str
    monthlySalary : int
    noOfPrj : int
    yearlyBonus : float
    yearlySalary : int
    prjEval : Literal["Very good","good","started"]
    summary : str

def calculateBonus(state : EmployeeDetails):
    prjCompleted = state["noOfPrj"]
    monthSal = state["monthlySalary"]
    if prjCompleted > 4 :
        bonus = monthSal * 5 /100
    elif prjCompleted >1 :
        bonus = monthSal * 3 /100
    else :
        bonus = monthSal * 1 /100
    yearlyBonus = 12 * bonus

    return {"yearlyBonus":yearlyBonus}

def calculateYearSalary(state : EmployeeDetails):
    monthSal = state["monthlySalary"]
    
    return {"yearlySalary":12 * monthSal}

def projectEvaluation(state : EmployeeDetails):
    prjCompl = state["noOfPrj"]
    if prjCompl > 4:
        eval = "Very good"
    elif prjCompl > 2:
        eval = "Good"
    else:
        eval = "started"
    
    return {"prjEval":eval}


def summary(state : EmployeeDetails):
    summary = f"Emploee {state['name']} has completed {state['noOfPrj']} projects successfully and has montly salary {state['monthlySalary']}.So, his salary per anum is {state['yearlySalary']} and bonus is {state['yearlyBonus']}."
    return {"summary":summary}

graph = StateGraph(state_schema=EmployeeDetails)
#get nodes 
graph.add_node("calculateBonus",calculateBonus)
graph.add_node("calculateYearSalary",calculateYearSalary)
graph.add_node("projectEvaluation",projectEvaluation)
graph.add_node("summary",summary)

# add edges
graph.add_edge(START, "calculateBonus")
graph.add_edge(START, "calculateYearSalary")
graph.add_edge(START, "projectEvaluation")
graph.add_edge("calculateBonus", "summary")
graph.add_edge("calculateYearSalary", "summary")
graph.add_edge("projectEvaluation", "summary")
graph.add_edge("summary", END)

workout = graph.compile()

final_state = workout.invoke({"name":"Vikram","monthlySalary":25000,"noOfPrj":4})
print(final_state)







In [ ]:
from IPython.display import Image
Image(workout.get_graph().draw_mermaid_png())

# LLM based parallel flow

START -> doc load essay <br>
-> evaluate analysis
-> evaluate language
-> evaluate thought

Find evaluation
END

In [2]:
# get llm
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-20b")


In [3]:
# evaluation schema
from pydantic import BaseModel,Field

class EvaluationSchema(BaseModel):
    feedback : str = Field(description="Detailed feedback for essay")
    score : int = Field(description="Score for essay out of 10",ge=0,le=10)

In [4]:
# structured llm
llm_structured = llm.with_structured_output(schema=EvaluationSchema)

In [5]:
# get the state
from typing import TypedDict,Annotated
from operator import add

class EssayDetails(TypedDict):
    essayPath : str
    essay : str
    lang_feedback : str
    analysis_feedback : str
    thought_feedback : str
    overall_feedback : str
    individual_score : Annotated[list[int],add]

In [6]:
# load the doc and get the essay content
from langchain_community.document_loaders import TextLoader
def loadEssay(state : EssayDetails):
  loader = TextLoader(state["essayPath"])
  essayDoc = loader.load()
  essay = essayDoc[0].page_content
  return {
    state["essay"] : essay
  }

/var/folders/ql/ds02c6jn4kzbc08btxntjvhr0000gn/T/ipykernel_5020/3036501931.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/langchain_langgraph/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# language evaluation
def evalLanguage(state : EssayDetails):
    prompt = f"Evaluate the language quality of given essay.Please give feed back and assign score out of 10.{state['essay']}"
    output = llm_structured.invoke(prompt)
    return {
        "lang_feedback" : output.feedback,
        "individual_score" : [output.score]
    }

In [8]:
# analysis evaluation
def evalLanguage(state : EssayDetails):
    prompt = f"Evaluate the analysis quality of given essay.Please give feed back and assign score out of 10.{state['essay']}"
    output = llm_structured.invoke(prompt)
    return {
        "analysis_feedback" : output.feedback,
        "individual_score" : [output.score]
    }

In [9]:
# analysis evaluation
def evalThought(state : EssayDetails):
    prompt = f"Evaluate the thought quality of given essay.Please give feed back and assign score out of 10.{state['essay']}"
    output = llm_structured.invoke(prompt)
    return {
        "thought_feedback" : output.feedback,
        "individual_score" : [output.score]
    }

In [10]:
# overall summary
def overallEvaluation(state : EssayDetails):
    prompt = f"Evaluate the overall sumamry using  feedback {state['lang_feedback']},{state['analysis_feedback']} and {state['thought_feedback']}"

    overall_feedback = llm.invoke(prompt)
    score = sum(state['individual_score'])/len(state['individual_score'])
    return {
        "thought_feedback" : overall_feedback,
        "individual_score" : score
    }

In [ ]:
# define graph now
from langgraph.graph import StateGraph, START,END

"/Users/sanvijanvi/Documents/SJ_HomeWork/essay.txt"